# Distil the teacher into the 0.8B: SFT on every train suite

One notebook for each round of distillation, top to bottom. The student learns from the teacher's sessions on every train suite there is, generated and real, so it reads real receipts without losing anything else:

| teacher run | train suite | cases |
|---|---|---|
| `teacher-train` | `train`: generated receipts, order emails, bank and wallet notifications, notes and non-spending messages in seven languages, for 210 users (`make_suite.py --name train --per-locale 300 --users-per-locale 30 --seed 1 --novel-share 0.5`) | 2,100 |
| `teacher-train-2` | `train-2`: the same mix, new cases for 420 new users (`--name train-2 --per-locale 600 --users-per-locale 60 --seed 4 --novel-share 0.5`) | 4,200 |
| `teacher-train-receipts` | `train-receipts`: real till receipts read with macOS's OCR, each twice (731 receipts) | 1,462 |

Half the generated users have their merchants and item names invented for them, so the student learns to read a receipt rather than recognise a shop. The real receipts come from other shops than `dev-receipts`. Never `dev`, `dev-novel`, `dev-receipts` or `holdout`: `sft_data.py` refuses the frozen suites, and `real_receipts.py` keeps every shop of `dev-receipts` out of `train-receipts`.

**Run `real_receipts_eval.ipynb` first.** It builds the receipt suites, audits their gold, and scores the shipped model, the untuned 0.8B and the teacher on `dev-receipts`. This notebook reads those suites and stops if they aren't there.

1. **The teacher on the train suites.** `Qwen/Qwen3.8-27B-FP8` on vLLM, with the settings of its `dev` baseline (`qwen3.8-27b-fp8-v5`). A run already finished is reused. Then the teacher on `dev-novel` and `dev-receipts`, if their rows are missing.
2. **The training pairs.** `sft_data.py` takes every run into one file. It keeps the sessions whose case graded `record_all`, rebuilds every request the agent sent in them, and renders each as a (prompt, completion) pair with Qwen3.5-0.8B's chat template, thinking off: tool calls, the end of the tool phase (an empty reply) and the answer JSON. The sessions that drafted a wrong record add their answer with its confidence set to `DOUBT_CONFIDENCE`, only that number trained on. A train case whose content a frozen suite repeats word for word is left out (`frozen_content` in the stats).
3. **SFT.** vLLM is stopped and `train_sft.py` trains a fresh LoRA from the base model: every linear layer including the Gated DeltaNet projections, loss on the completion only, averaged per token.
4. **Merge.** `merge_lora.py` adds the adapter into a copy of the original checkpoint, so the result has the base model's layout (`Qwen3_5ForConditionalGeneration`) that llama.cpp's converter reads.
5. **Score it unquantized.** The merged model at bf16 on vLLM over `dev`, `dev-novel` and `dev-receipts`, each beside the earlier rounds, the shipped model and the teacher. For a GGUF row on `dev-receipts` like the shipped one's, add the merged model's quantized file to `MODELS` in `real_receipts_eval.ipynb`.

| | teacher pass | SFT |
|---|---|---|
| model | `Qwen/Qwen3.8-27B-FP8` | `Qwen/Qwen3.5-0.8B` + a fresh LoRA |
| server / trainer | vLLM, 16 cases in flight (`--max-num-seqs 32`), thinking off, 10 tool rounds / 800 / 1,500 tokens | `train_sft.py`, bf16, rank 16, 1 epoch, lr 2e-4, 8 sequences a step, `--max-len 8192`, loss per token |

The teacher needs a GPU with more than 40 GB; training the student needs bf16. The notebook sees when the teacher's work is done, and then never starts the teacher: it only trains and scores the student. It stops early if a teacher run is still needed and the GPU is too small.

**Every long step resumes.** If the session disconnects, run the cells again from the top: a teacher run resumes from its last finished case, training from its last checkpoint, and finished steps are skipped. The merged model is what `grpo.ipynb` (`SFT_ROUND`) continues and `baseline_gguf.ipynb` (`SOURCE`) quantizes.

The notebook clones `REPO_URL` at `BRANCH`, [kurexi/cost](https://github.com/kurexi/cost) by default; for a private fork, add a secret named `GITHUB_TOKEN` with read access to it. `RECORDS` is the folder, inside the clone, that holds `runs/` and `RESULTS.md`; empty by default, which keeps them in the agent's `eval/` folder, where git ignores them. The receipts, the receipt suites and the transcripts stay out of git.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None, env=None):
    process = subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args[:3])


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "claude/modest-brahmagupta-hpbdfb"
RECORDS = ""
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

**The receipt suites.** `TRAIN_SUITE` and `EVAL_SUITE` are the suites `real_receipts_eval.ipynb` built. This notebook only reads them.

**The round.** `TEACHER_RUNS` lists the teacher runs the pairs are built from. Generated suites are regenerated and checked against their fingerprints; the real-receipt suite is marked `real`. Every train suite is used: the generated `train` (2,100 cases) and `train-2` (4,200), and the real receipts (1,462), so the student learns receipts without losing emails, notifications, notes and the other languages. A run already finished is reused. `ROUND` names everything this notebook makes, so a round never overwrites another: the adapter, the merged model (`qwen3.5-0.8b-sft-r6`) and its runs. `PREVIOUS`, `EARLIER` and `SHIPPED` are what it is compared with. A run of them that isn't present is left out of the comparison.

In [ ]:
TRAIN_SUITE = "train-receipts"
EVAL_SUITE = "dev-receipts"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"

TEACHER = "Qwen/Qwen3.8-27B-FP8"
TEACHER_RUNS = {
    "teacher-train": {"suite": "train", "fingerprint": "a5437ef135b8f48f", "cases": 2100,
                      "args": ["--per-locale", "300", "--users-per-locale", "30", "--seed", "1", "--novel-share", "0.5"]},
    "teacher-train-2": {"suite": "train-2", "fingerprint": "9c208ad58ed4d37a", "cases": 4200,
                        "args": ["--per-locale", "600", "--users-per-locale", "60", "--seed", "4", "--novel-share", "0.5"]},
    f"teacher-{TRAIN_SUITE}": {"suite": TRAIN_SUITE, "real": True},
}
STUDENT = "Qwen/Qwen3.5-0.8B"
ROUND = "r6"
PREVIOUS = "qwen3.5-0.8b-sft-r5"
EARLIER = "qwen3.5-0.8b-sft-r3"
SHIPPED = "qwen3.5-0.8b-grpo-r2"
LOSS = "token"
DOUBT_CONFIDENCE = 0.5
NAME = f"qwen3.5-0.8b-sft-{ROUND}"
EPOCHS = 1
RANK = 16
LR = 2e-4
ACCUM = 8
MAX_LEN = 8192
print(NAME)

## Storage

The teacher's runs under `runs/` are links into storage that outlives the session, so `run.py` appends each finished case there and a rerun resumes from it.

In [ ]:
import json
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
SUITES_KEPT = f"{DRIVE}/suites"
for folder in (SUITES_KEPT, f"{DRIVE}/runs"):
    os.makedirs(folder, exist_ok=True)


def keep_on_drive(run):
    kept, local = f"{DRIVE}/runs/{run}", f"{RECORDS_DIR}/runs/{run}"
    os.makedirs(kept, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local) and not os.listdir(kept):
        shutil.copytree(local, kept, dirs_exist_ok=True)
    sh(["rm", "-rf", local])
    os.symlink(kept, local)


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases


def has_transcripts(folder, cases):
    transcripts = f"{folder}/transcripts.jsonl"
    return os.path.exists(transcripts) and sum(1 for _ in open(transcripts)) >= cases

TEACHER_DIRS = {run: f"{DRIVE}/runs/{run}" for run in TEACHER_RUNS}
TEACHER_NOVEL_RUN = f"{TEACHER_NAME}-dev-novel"
TEACHER_REAL_RUN = f"{TEACHER_NAME}-{EVAL_SUITE}"
PAIRS = f"{DRIVE}/sft/{'+'.join(TEACHER_RUNS)}" + (f"-doubt{DOUBT_CONFIDENCE:g}" if DOUBT_CONFIDENCE is not None else "") + ".jsonl"
ADAPTER = f"{DRIVE}/sft/{NAME}"
MERGED = f"{DRIVE}/models/{NAME}"
for folder in (os.path.dirname(PAIRS), ADAPTER, os.path.dirname(MERGED)):
    os.makedirs(folder, exist_ok=True)

## The receipt suites

Copied from where `real_receipts_eval.ipynb` left them. The cell stops if they aren't there. If they were rebuilt after the teacher's runs on them, `run.py` refuses to resume on the new fingerprint, so a run is never graded against other cases than it ran on.

In [ ]:
os.chdir(EVAL)
missing = [name for name in (TRAIN_SUITE, EVAL_SUITE) if not os.path.exists(f"{SUITES_KEPT}/{name}/meta.json")]
if missing:
    raise SystemExit(f"{missing} aren't in {SUITES_KEPT}: run real_receipts_eval.ipynb first. It builds them from "
                     "the receipts, audits their gold and scores the baseline on them")
SUITE_CASES = {}
for name in (TRAIN_SUITE, EVAL_SUITE):
    shutil.copytree(f"{SUITES_KEPT}/{name}", f"suites/{name}", dirs_exist_ok=True)
    with open(f"suites/{name}/meta.json") as f:
        meta = json.load(f)
    SUITE_CASES[name] = meta["cases"]
    print(name, {k: meta[k] for k in ("cases", "users", "receipts", "fingerprint")})

## The generated train suites, and what is still to run

Each generated suite is regenerated from its seed and checked against its fingerprint, so it is the same suite every time. Then the cell checks that each finished teacher run still has its `transcripts.jsonl` (the pairs are built from them), and says which teacher runs are still to do: none, once the pairs are built and the teacher has its `dev-novel` and `dev-receipts` rows.

In [ ]:
os.chdir(EVAL)
for run, spec in TEACHER_RUNS.items():
    if spec.get("real"):
        spec["cases"] = SUITE_CASES[spec["suite"]]
        continue
    subprocess.run(["python3", "make_suite.py", "--name", spec["suite"], *spec["args"]], check=True, capture_output=True)
    with open(f"suites/{spec['suite']}/meta.json") as f:
        suite = json.load(f)
    print(spec["suite"], suite["cases"], "cases, fingerprint", suite["fingerprint"])
    if suite["fingerprint"] != spec["fingerprint"] or suite["cases"] != spec["cases"]:
        raise SystemExit(f"{spec['suite']} doesn't regenerate as it did: the generator changed, so this is a new suite")

for run in (*TEACHER_RUNS, TEACHER_NOVEL_RUN, TEACHER_REAL_RUN):
    keep_on_drive(run)
if os.path.exists(PAIRS):
    with open(PAIRS.replace(".jsonl", ".stats.json")) as f:
        short = [r["run"] for r in json.load(f).get("runs", []) if r["sessions"] < TEACHER_RUNS[r["run"]]["cases"]]
    if short:
        raise SystemExit(f"{PAIRS} was built without the sessions of {short}: delete it and its .stats.json, "
                         "then run this cell again")
LOST = [] if os.path.exists(PAIRS) else [
    run for run, spec in TEACHER_RUNS.items()
    if finished(TEACHER_DIRS[run], spec["cases"]) and not has_transcripts(TEACHER_DIRS[run], spec["cases"])]
if LOST:
    raise SystemExit(f"{LOST} have their results but not their transcripts.jsonl, and the training pairs are "
                     f"built from the transcripts. Look in {DRIVE}/runs on drive.google.com for a second folder of the "
                     "same name, which a dropped connection can leave, and move its transcripts.jsonl into this one")
gpu_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()[0])
TEACHER_FITS = gpu_mib >= 39000
NEEDS_TRAIN_RUNS = [] if os.path.exists(PAIRS) else [
    run for run, spec in TEACHER_RUNS.items() if not finished(TEACHER_DIRS[run], spec["cases"])]
NEEDS_EVAL_RUNS = [(run, suite) for run, suite, cases in ((TEACHER_NOVEL_RUN, "dev-novel", 280),
                                                          (TEACHER_REAL_RUN, EVAL_SUITE, SUITE_CASES[EVAL_SUITE]))
                   if not finished(f"{DRIVE}/runs/{run}", cases)]
if NEEDS_TRAIN_RUNS and not TEACHER_FITS:
    raise SystemExit(f"the teacher runs {NEEDS_TRAIN_RUNS} aren't finished and this GPU has {gpu_mib} MiB: the teacher needs 40 GB or more")
SERVE_TEACHER = TEACHER_FITS and (bool(NEEDS_TRAIN_RUNS) or bool(NEEDS_EVAL_RUNS))
for run in TEACHER_RUNS:
    print(f"teacher run {run}:", "to run" if run in NEEDS_TRAIN_RUNS else "done")
for run, suite in NEEDS_EVAL_RUNS:
    print(f"teacher on {suite}:", "to run" if TEACHER_FITS else "missing, and this GPU can't host the teacher")

## vLLM

vLLM goes in its own virtual environment, so its pinned `torch` and `transformers` can't clash with the training half. Same flags as the teacher's `dev` run: the checkpoint is FP8 already, 16,384 tokens per sequence, `--max-num-seqs 32`, the `qwen3_coder` tool parser and the `qwen3` reasoning parser, served as `qwen`. Logs go to `/content/vllm.log`. vLLM is installed either way: the student is scored on it too.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
    sh(["pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def wait_for(process, log_path, name, limit_seconds=2400):
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"{name} exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"{name} is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > limit_seconds:
            raise SystemExit(f"{name} isn't up after {limit_seconds} s:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)


def serve(model, extra=""):
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_for(process, log_path, "vLLM")


RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]


def check_server():
    os.chdir(EVAL)
    probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                           capture_output=True, text=True)
    print(probe.stdout, probe.stderr)
    verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
    for required in ("tool round (tool_choice auto)", "full bounded schema"):
        if not verdicts.get(required, "").startswith("accepted"):
            raise SystemExit(f"the server refused the {required}: not a fair run, stop here")

## The teacher, with the harness checked first

On `train-receipts`: `oracle` must grade 1.000 and `silent` 0, and the server must accept the tool round and the full bounded schema. Then come the train runs still to do, and the teacher's evaluation rows if they are missing. Every run resumes after a disconnect. The grade at the end of each says how many sessions are right enough to train on (`record_all`).

In [ ]:
os.chdir(EVAL)
if SERVE_TEACHER:
    serve(TEACHER)
    for name, backend in (("check-oracle", "oracle"), ("check-silent", "silent")):
        graded = subprocess.run(["python3", "run.py", "--suite", TRAIN_SUITE, "--name", name, "--backend", backend,
                                 "--limit", "70", "--restart"], capture_output=True, text=True).stdout
        print(name, [line for line in graded.splitlines() if line.startswith("record_core")])
    check_server()
    for run in NEEDS_TRAIN_RUNS:
        sh(["python3", "run.py", "--suite", TEACHER_RUNS[run]["suite"], "--name", run, *RUN_ARGS])
    for run, suite in NEEDS_EVAL_RUNS:
        sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
    stop_servers()
else:
    print("the teacher isn't needed" if not NEEDS_EVAL_RUNS else "the teacher's evaluation rows are missing: the teacher needs a GPU with 40 GB or more")

## The training pairs

Training runs in the notebook's own Python: `transformers` new enough for Qwen3.5, `peft`, and the `flash-linear-attention` kernels for the DeltaNet layers (without them the delta rule runs on a slow reference implementation). The preinstalled `torchao` is removed: training doesn't use it, and `peft` refuses to attach a LoRA while an older version than it supports is installed. The student's checkpoint is downloaded once to local disk.

`sft_data.py` checks its template renderer against `transformers`' `apply_chat_template` on every request of 20 sessions before it writes anything, counts tokens, and drops any pair over `MAX_LEN` instead of letting the trainer cut the instructions off its start. It stops if a run's transcripts are missing. Its counts are written beside the pairs: the totals, then each run's sessions, `kept`, `doubted` and `frozen_content`. Every run should show as many sessions as it has cases. Each pairs file is built once and reused.

In [ ]:
!pip install -q -U transformers peft flash-linear-attention
!pip uninstall -q -y torchao
from huggingface_hub import snapshot_download

BASE = "/content/models/qwen35-08b"
snapshot_download(STUDENT, local_dir=BASE)
%cd {EVAL}
if not os.path.exists(PAIRS):
    sh(["python3", "sft_data.py", *TEACHER_RUNS, "--template", BASE, "--tokenizer", BASE,
        "--verify-template", "--max-len", str(MAX_LEN), "--out", PAIRS,
        *(["--doubt-confidence", str(DOUBT_CONFIDENCE)] if DOUBT_CONFIDENCE is not None else [])])
print(open(PAIRS.replace(".jsonl", ".stats.json")).read())

## SFT

A fresh LoRA from the base model on all the pairs. One sequence per pair, eight to an optimizer step, the loss on the completion only, averaged as `LOSS` says (`train_args.json` beside the adapter records it). It saves a checkpoint every 50 steps and resumes from it when rerun. Each step prints its loss by kind.

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    sh(["python3", "train_sft.py", "--model", BASE, "--data", PAIRS,
        "--out", ADAPTER, "--epochs", str(EPOCHS), "--rank", str(RANK), "--lr", str(LR),
        "--accum", str(ACCUM), "--max-len", str(MAX_LEN), "--loss", LOSS, "--save-every", "50"])
!tail -n 3 {ADAPTER}/train_log.jsonl

## Merge

The adapter is added into a copy of the downloaded checkpoint, which keeps its layout, its tokenizer and its chat template. The merged folder is what `baseline_gguf.ipynb` converts to a GGUF.

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{MERGED}/merge.json"):
    sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", ADAPTER, "--out", MERGED])
!ls -la {MERGED}

## The merged model on `dev`, `dev-novel` and `dev-receipts`

At bf16 on vLLM. Each suite is compared with the baseline from `real_receipts_eval.ipynb` and the earlier rounds; the first run present is the baseline. On `dev-receipts` that is the shipped file (Q8_0, which costs next to nothing against bf16 on `dev`). Each run also prints the share of the drafts it would auto-approve that are right (the approval precision). `dev-receipts` says whether real receipts got better, and `dev` and `dev-novel` say whether anything else got worse. The audit then shows the cases the new model still gets wrong.

In [ ]:
serve(MERGED, "--dtype bfloat16")
os.chdir(EVAL)
SCORED = {"dev": f"{NAME}-bf16", "dev-novel": f"{NAME}-bf16-dev-novel", EVAL_SUITE: f"{NAME}-bf16-{EVAL_SUITE}"}
for suite, run in SCORED.items():
    keep_on_drive(run)
    sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
stop_servers()
for run in (f"{SHIPPED}-q8_0-{EVAL_SUITE}", f"qwen3.5-0.8b-fp16-{EVAL_SUITE}"):
    keep_on_drive(run)
LINEUPS = {
    "dev": [f"{EARLIER}-bf16", f"{PREVIOUS}-bf16", f"{SHIPPED}-bf16", SCORED["dev"], TEACHER_NAME],
    "dev-novel": [f"{EARLIER}-bf16-dev-novel", f"{PREVIOUS}-bf16-dev-novel", f"{SHIPPED}-bf16-dev-novel",
                  SCORED["dev-novel"], TEACHER_NOVEL_RUN],
    EVAL_SUITE: [f"{SHIPPED}-q8_0-{EVAL_SUITE}", f"qwen3.5-0.8b-fp16-{EVAL_SUITE}", SCORED[EVAL_SUITE], TEACHER_REAL_RUN],
}
for suite, lineup in LINEUPS.items():
    cases = SUITE_CASES.get(suite, 280)
    present = [run for run in lineup if finished(f"{RECORDS_DIR}/runs/{run}", cases)]
    print(f"\n#### {suite}: {present}")
    if len(present) >= 2:
        sh(["python3", "compare.py", *present])
    for run in present:
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            per_case = json.load(f)["per_case"]
        approved = [c for c in per_case if c["would_approve"]]
        print(f"{run}: approval precision {sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} "
              f"over {len(approved)} approved")
present = [run for run in (SCORED[EVAL_SUITE], f"{SHIPPED}-q8_0-{EVAL_SUITE}", TEACHER_REAL_RUN)
           if finished(f"{RECORDS_DIR}/runs/{run}", SUITE_CASES[EVAL_SUITE])]
if present:
    sh(["python3", "audit.py", *present, "--limit", "15"])
sh(["python3", "board.py", "--write"])

## Take the results home

What is worth keeping: for each run, its `meta.json`, `results.jsonl` and `grade.json`, plus the pair counts, the adapter's config and train log, the merge report, the suites' `meta.json` and `RESULTS.md`. The receipts, the suites and the transcripts aren't in it.

In [ ]:
OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
for run, folder in TEACHER_DIRS.items():
    os.makedirs(f"{OUT}/runs/{run}")
    for name in ("meta.json", "results.jsonl", "grade.json"):
        shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
os.makedirs(f"{OUT}/sft/{NAME}")
for source in (PAIRS.replace(".jsonl", ".stats.json"), f"{ADAPTER}/adapter_config.json", f"{ADAPTER}/train_args.json",
               f"{ADAPTER}/train_log.jsonl", f"{MERGED}/merge.json"):
    shutil.copy(source, f"{OUT}/sft/{NAME}/")
for run in (TEACHER_NOVEL_RUN, TEACHER_REAL_RUN, *SCORED.values()):
    folder = f"{RECORDS_DIR}/runs/{run}"
    if finished(folder, 1):
        os.makedirs(f"{OUT}/runs/{run}")
        for name in ("meta.json", "results.jsonl", "grade.json"):
            if os.path.exists(f"{folder}/{name}"):
                shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
for name in (TRAIN_SUITE, EVAL_SUITE):
    os.makedirs(f"{OUT}/suites/{name}")
    shutil.copy(f"suites/{name}/meta.json", f"{OUT}/suites/{name}/")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{NAME}.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{NAME}.zip")